# AI Engineer Pandas cleaning

In [1]:
import pandas as pd
data = {
    "model":     ["BERT", "XGBoost", None, "ResNet", "GPT-3", "BERT"],
    "accuracy":  [0.94, 0.91, 0.88, None, 0.96, 0.94],
    "loss":      [0.21, 0.34, 0.45, 0.28, None, 0.21],
    "epochs":    [10, 100, 50, None, 30, 10],
    "framework": ["PyTorch", "XGBoost", "TensorFlow", "PyTorch", "OpenAI", "PyTorch"]
}
df=pd.DataFrame(data)
print("Missing values before filling:")
print(df.isnull().sum())
null_val=df.isnull().sum()
df["accuracy"] = df["accuracy"].fillna(df["accuracy"].mean())
df["epochs"]   = df["epochs"].fillna(df["epochs"].median())
df["model"]    = df["model"].fillna(df["model"].mode()[0])
df["loss"]     = df["loss"].ffill()
print("\nMissing values after filling:\n")
print(df.isnull().sum())

# Optional: print final DataFrame
print("\nFinal DataFrame:\n")
print(df)


Missing values before filling:
model        1
accuracy     1
loss         1
epochs       1
framework    0
dtype: int64

Missing values after filling:

model        0
accuracy     0
loss         0
epochs       0
framework    0
dtype: int64

Final DataFrame:

     model  accuracy  loss  epochs   framework
0     BERT     0.940  0.21    10.0     PyTorch
1  XGBoost     0.910  0.34   100.0     XGBoost
2     BERT     0.880  0.45    50.0  TensorFlow
3   ResNet     0.926  0.28    30.0     PyTorch
4    GPT-3     0.960  0.28    30.0      OpenAI
5     BERT     0.940  0.21    10.0     PyTorch


In [2]:
# handling missing row value
# Shape before removing duplicates
print("Shape before:", df.shape)

# Check for duplicate rows
duplicates = df.duplicated()

print("\nDuplicate flags:\n")
print(duplicates)

# Print duplicate rows
print("\nDuplicate rows:\n")
print(df[duplicates])

# Remove duplicates
df = df.drop_duplicates()

# Shape after removing duplicates
print("\nShape after:", df.shape)

Shape before: (6, 5)

Duplicate flags:

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool

Duplicate rows:

  model  accuracy  loss  epochs framework
5  BERT      0.94  0.21    10.0   PyTorch

Shape after: (5, 5)


In [3]:
# strip
data = {
    "model":     ["  bert  ", "XGBOOST", "resnet", "  GPT-3 "],
    "accuracy":  ["0.94", "0.91", "0.88", "0.96"],
    "year":      ["2018", "2016", "2015", "2020"]
}
df=pd.DataFrame(data)
print(df.dtypes)
df["model"]=df["model"].str.strip()
df["model"]=df["model"].str.upper()
df["accuracy"]=df["accuracy"].astype(float)
df["year"]=df["year"].astype(int)
print(df.dtypes)

model       object
accuracy    object
year        object
dtype: object
model        object
accuracy    float64
year          int64
dtype: object


In [4]:
# label encoding and one hot encoding
data = {
    "framework": ["PyTorch", "Sklearn", "PyTorch", "TensorFlow", "Sklearn", "PyTorch"],
    "grade":     ["High", "Low", "High", "Medium", "Low", "High"]
}
df=pd.DataFrame(data)
grade_mapping = {"Low": 0, "Medium": 1, "High": 2}
df["grade_encoded"] = df["grade"].map(grade_mapping)

print("Label Encoded DataFrame:\n")
print(df)

# Step 3: One-Hot Encoding for framework
framework_dummies = pd.get_dummies(df["framework"], prefix="framework")

print("\nOne-Hot Encoded Framework:\n")
print(framework_dummies)

Label Encoded DataFrame:

    framework   grade  grade_encoded
0     PyTorch    High              2
1     Sklearn     Low              0
2     PyTorch    High              2
3  TensorFlow  Medium              1
4     Sklearn     Low              0
5     PyTorch    High              2

One-Hot Encoded Framework:

   framework_PyTorch  framework_Sklearn  framework_TensorFlow
0               True              False                 False
1              False               True                 False
2               True              False                 False
3              False              False                  True
4              False               True                 False
5               True              False                 False


In [8]:
# creating pipeline
import pandas as pd
import numpy as np

def clean_ml_dataset(df):
    df = df.copy()  # avoid modifying original
    
    # 1. Remove duplicates
    df = df.drop_duplicates()
    
    # 2. Separate column types
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns
    
    # 3. Fill missing numerical values with median
    for col in num_cols:
        df[col]=df[col].fillna(df[col].median())
    
    # 4. Fill missing categorical values with mode
    for col in cat_cols:
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode()[0])
    
    # 5. Strip whitespace from string columns
    for col in cat_cols:
        df[col] = df[col].astype(str).str.strip()
    
    return df
# Create messy data explicitly — don't rely on previous cell's data variable
df_messy = pd.DataFrame({
    "model":     ["BERT", "XGBoost", None, "ResNet", "GPT-3", "BERT"],
    "accuracy":  [0.94, 0.91, 0.88, None, 0.96, 0.94],
    "loss":      [0.21, 0.34, 0.45, 0.28, None, 0.21],
    "epochs":    [10, 100, 50, None, 30, 10],
    "framework": ["PyTorch", "XGBoost", "TensorFlow", "PyTorch", "OpenAI", "PyTorch"]
})
print("Before cleaning:")
print(df_messy.isnull().sum())

df_clean = clean_ml_dataset(df_messy)
print("\nAfter cleaning:")
print(df_clean.isnull().sum())
print(df_clean)

Before cleaning:
model        1
accuracy     1
loss         1
epochs       1
framework    0
dtype: int64

After cleaning:
model        0
accuracy     0
loss         0
epochs       0
framework    0
dtype: int64
     model  accuracy  loss  epochs   framework
0     BERT     0.940  0.21    10.0     PyTorch
1  XGBoost     0.910  0.34   100.0     XGBoost
2     BERT     0.880  0.45    50.0  TensorFlow
3   ResNet     0.925  0.28    40.0     PyTorch
4    GPT-3     0.960  0.31    30.0      OpenAI
